## 2 序列模型

### 2.1 理论计算题

给定字符序列 `"ababc"`，词汇表 $\{a,b,c\}$。采用一阶马尔可夫模型，使用拉普拉斯平滑（加 1 平滑）估计条件概率。

首先统计所有相邻转移（从 $t$ 到 $t+1$）的频数：

| 转移 | 次数 |
|------|------|
| a→b  | 2    |
| b→a  | 1    |
| b→c  | 1    |
| 其他 | 0    |

对于条件概率 $p(x' | b)$，分母为从状态 $b$ 出发的总转移次数加上词汇表大小 $V=3$（加 1 平滑）。从 $b$ 出发的原始总次数为 $1+1=2$（b→a 和 b→c），加平滑后分母为 $2+3=5$。

分子为对应转移的原始频数加 1：

- $p(a | b) = \frac{1+1}{2+3} = \frac{2}{5}$
- $p(c | b) = \frac{1+1}{2+3} = \frac{2}{5}$


### 2.2 编程题

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写，去除标点符号
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表，按频率降序分配ID
    word_counts = Counter(words)
    vocab = {word: idx for idx, (word, _) in enumerate(word_counts.most_common())}
    
    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    # 注意：要循环到 len(words) - n + 1，确保最后一个窗口也被包含
    for i in range(len(words) - n + 1):
        features.append([words[i+j] for j in range(n)])
        # 如果 i+n 在范围内，取下一个词作为标签；否则补 None
        if i + n < len(words):
            labels.append(words[i+n])
        else:
            labels.append(None)
    
    return vocab, (features, labels)

# 测试
vocab, (feat, lab) = preprocess_text("The time machine", 2)
print("词汇表:", vocab)
print("特征:", feat)
print("标签:", lab)

词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征: [['the', 'time'], ['time', 'machine']]
标签: ['machine', None]


## 3 循环神经网络

### 3.1 理论计算题

线性 RNN（无偏置）：
$$
h_t = W_{hh} h_{t-1} + W_{hx} x_t, \quad o_t = W_{oh} h_t
$$
损失函数：
$$
L = \frac{1}{2} \sum_{t=1}^{T} (o_t - y_t)^2
$$

通过时间反向传播（BPTT），损失对 $W_{hh}$ 的梯度为：
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial o_t} \frac{\partial o_t}{\partial h_t} \frac{\partial h_t}{\partial W_{hh}}
$$
而 $\frac{\partial L}{\partial o_t} = o_t - y_t$，$\frac{\partial o_t}{\partial h_t} = W_{oh}$。

关键是要展开 $\frac{\partial h_t}{\partial W_{hh}}$ 对之前所有时间步的依赖：
$$
\frac{\partial h_t}{\partial W_{hh}} = \sum_{k=1}^{t} \left( \prod_{i=k+1}^{t} W_{hh} \right) \frac{\partial h_k}{\partial W_{hh}} \quad (\text{局部})
$$
更精确地，完整梯度为：
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} (o_t - y_t)^\top W_{oh} \sum_{k=1}^{t} \left( \prod_{i=k+1}^{t} W_{hh} \right) h_{k-1}^\top
$$
（这里将 $h_{k-1}$ 视为常数，对 $W_{hh}$ 的导数来自 $h_t$ 对 $W_{hh}$ 的线性依赖。）

若令 $\delta_t = (o_t - y_t)^\top W_{oh}$，则梯度可写为：
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t \sum_{k=1}^{t} \left( W_{hh} \right)^{t-k} h_{k-1}^\top
$$

**梯度消失 / 爆炸条件**：  
当 $W_{hh}$ 的特征值的模大于 1 时，$(W_{hh})^{t-k}$ 会指数增长，导致梯度爆炸；当特征值的模小于 1 时，梯度指数衰减，导致梯度消失。若特征值模等于 1，梯度可能保持稳定（但线性 RNN 通常不稳定）。

### 3.2 编程题

In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hh, W_hx, b_h):
    """
    x_t: (batch_size, input_size)
    h_prev: (batch_size, hidden_size)
    W_hh: (hidden_size, hidden_size)
    W_hx: (input_size, hidden_size)
    b_h: (hidden_size,)
    """
    h_t = np.tanh(np.dot(h_prev, W_hh) + np.dot(x_t, W_hx) + b_h)
    cache = (x_t, h_prev, W_hh, W_hx, b_h, h_t)
    return h_t, cache

def rnn_cell_backward(dh_next, cache):
    """
    dh_next: (batch_size, hidden_size)  损失对 h_t 的梯度
    返回: dx_t, dh_prev, dW_hh, dW_hx, db_h
    """
    x_t, h_prev, W_hh, W_hx, b_h, h_t = cache
    # tanh 导数: 1 - h_t^2
    dtanh = dh_next * (1 - h_t**2)   # (batch, hidden)
    
    # 输入和隐藏状态的梯度
    dx_t = np.dot(dtanh, W_hx.T)      # (batch, input_size)
    dh_prev = np.dot(dtanh, W_hh.T)   # (batch, hidden_size)
    
    # 权重的梯度（注意累加 batch 维度）
    dW_hx = np.dot(x_t.T, dtanh)      # (input_size, hidden_size)
    dW_hh = np.dot(h_prev.T, dtanh)   # (hidden_size, hidden_size)
    db_h = np.sum(dtanh, axis=0)      # (hidden_size,)
    
    return dx_t, dh_prev, dW_hh, dW_hx, db_h

In [3]:
# 测试 rnn_cell_forward 和 rnn_cell_backward
np.random.seed(42)
batch_size, input_size, hidden_size = 4, 3, 5
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hh = np.random.randn(hidden_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size)
b_h = np.random.randn(hidden_size)

# 前向
h_t, cache = rnn_cell_forward(x_t, h_prev, W_hh, W_hx, b_h)
print("h_t shape:", h_t.shape)
print("h_t first row:\n", h_t[0, :5])  # 只打印部分

# 反向（假设上游梯度为随机）
dh_next = np.random.randn(batch_size, hidden_size)
dx_t, dh_prev, dW_hh, dW_hx, db_h = rnn_cell_backward(dh_next, cache)
print("dx_t shape:", dx_t.shape)
print("dh_prev shape:", dh_prev.shape)
print("dW_hh shape:", dW_hh.shape)
print("dW_hx shape:", dW_hx.shape)
print("db_h shape:", db_h.shape)
print("dW_hh first 3x3:\n", dW_hh[:3, :3])

h_t shape: (4, 5)
h_t first row:
 [ 0.99981729  0.99947312 -0.33242126 -0.7450054   0.98868817]
dx_t shape: (4, 3)
dh_prev shape: (4, 5)
dW_hh shape: (5, 5)
dW_hx shape: (3, 5)
db_h shape: (5,)
dW_hh first 3x3:
 [[-1.34033033e-01 -1.99681022e-03 -4.67832158e-01]
 [ 1.73505781e-01  2.98623729e-02  3.17545096e+00]
 [ 7.81130477e-02  1.22351784e-02  3.06098228e+00]]


## 4 高级循环神经网络

### 4.1 理论计算题

深度双向 RNN，共 $L$ 层，每层隐藏单元数为 $H$，输入维度 $D$，输出维度 $O$（忽略输出层前的投影，即不计算输出层的参数）。每层包含前向和后向两个独立的 RNN，每个 RNN 有输入到隐藏的权重矩阵、隐藏到隐藏的权重矩阵和偏置。

- **第 1 层**：输入维度为 $D$。  
  前向 RNN 参数：$W_{hx}^{(1,f)} \in \mathbb{R}^{D \times H}$，$W_{hh}^{(1,f)} \in \mathbb{R}^{H \times H}$，$b_h^{(1,f)} \in \mathbb{R}^{H}$。  
  后向 RNN 同理，参数数量相同。  
  所以第 1 层总参数：$2 \times (D \cdot H + H^2 + H)$。

- **第 $l$ 层（$l>1$）**：输入为上一层的输出拼接（双向），维度为 $2H$。  
  前向和后向各有一个 RNN，每个的参数为：$(2H) \cdot H + H^2 + H$。  
  因此第 $l$ 层参数：$2 \times (2H \cdot H + H^2 + H) = 2 \times (3H^2 + H)$。

总参数数量：
$$
\text{总参数} = 2(DH + H^2 + H) + \sum_{l=2}^{L} 2(2H \cdot H + H^2 + H)
$$
化简：
$$
= 2(DH + H^2 + H) + 2(L-1)(3H^2 + H)
$$
若 $L=1$，则为 $2(DH + H^2 + H)$。  
注意：偏置项在每个 RNN 中只有一个向量，我们已经计入。

### 4.2 编程题

In [4]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers, batch_first=False, bidirectional=True)
        # 输出维度为 hidden_dim * 2，因为双向拼接
    
    def forward(self, X):
        """
        X: (seq_len, batch, input_dim)
        返回:
          outputs: (seq_len, batch, 2*hidden_dim)  每个时间步拼接前后向
          final_state: (batch, 2*hidden_dim)       最终时间步的拼接（最后一步的前向 + 最后一步的后向）
        """
        outputs, h_n = self.rnn(X)  # h_n: (num_layers*2, batch, hidden_dim)
        # 取最后一层的前向和后向隐藏状态
        # h_n 索引: 0..num_layers-1 为前向，num_layers..2*num_layers-1 为后向
        forward_last = h_n[-2]   # 前向最后一层
        backward_last = h_n[-1]  # 后向最后一层
        final_state = torch.cat([forward_last, backward_last], dim=-1)  # (batch, 2*hidden_dim)
        return outputs, final_state

In [5]:
# 测试 BiRNNEncoder
torch.manual_seed(42)
input_dim, hidden_dim, num_layers = 4, 6, 1
seq_len, batch = 7, 3
X = torch.randn(seq_len, batch, input_dim)

model = BiRNNEncoder(input_dim, hidden_dim, num_layers)
outputs, final_state = model(X)

print("outputs shape:", outputs.shape)          # (seq_len, batch, 2*hidden_dim)
print("final_state shape:", final_state.shape) # (batch, 2*hidden_dim)
print("final_state first sample:\n", final_state[0, :8])  # 部分数值

outputs shape: torch.Size([7, 3, 12])
final_state shape: torch.Size([3, 12])
final_state first sample:
 tensor([-0.0934,  0.6542,  0.0473, -0.4285,  0.4769, -0.3320,  0.3819,  0.8231],
       grad_fn=<SliceBackward0>)


## 5 嵌入向量

### 5.1 理论计算题

Skip-gram 负采样损失函数（对单个中心词 $w_c$ 和上下文词 $w_o$，采样 $K$ 个负样本）：

给定词向量 $\mathbf{v}_c$（中心词输入向量）和 $\mathbf{u}_o$（上下文词输出向量），负样本词向量 $\mathbf{u}_{n_k}$（从噪声分布 $P_n(w)$ 中采样）。目标是最小化负对数似然：

$$
\mathcal{L} = - \log \sigma(\mathbf{v}_c^\top \mathbf{u}_o) - \sum_{k=1}^{K} \log \sigma(-\mathbf{v}_c^\top \mathbf{u}_{n_k})
$$

其中 $\sigma(x) = \frac{1}{1+e^{-x}}$。  
负样本从噪声分布 $P_n(w)$ 中采样，常见为 unigram 分布的 $3/4$ 次方（如 Word2Vec 中的做法）。

### 5.2 编程题

In [6]:
import numpy as np

def cbow_forward(context_indices, center_index, W, W_out):
    """
    context_indices: list of lists, each list of context word indices (shape: (batch, context_size))
    center_index: 目标中心词索引 (batch,)
    W: (V, d) 输入嵌入矩阵
    W_out: (d, V) 输出权重矩阵
    返回: loss (标量)
    """
    batch_size = len(context_indices)
    context_size = len(context_indices[0])
    V, d = W.shape
    
    # 1. 取上下文词向量并平均
    avg_vectors = []
    for sample in context_indices:
        vecs = [W[idx] for idx in sample]  # 每个 (d,)
        avg = np.mean(vecs, axis=0)        # (d,)
        avg_vectors.append(avg)
    hidden = np.array(avg_vectors)         # (batch, d)
    
    # 2. 输出得分
    scores = np.dot(hidden, W_out)         # (batch, V)
    
    # 3. softmax + 交叉熵损失
    exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))  # 数值稳定
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)       # (batch, V)
    
    # 4. 损失：对每个样本取目标词的概率的负对数
    loss = -np.mean(np.log(probs[np.arange(batch_size), center_index]))
    return loss

In [7]:
# 测试 cbow_forward
np.random.seed(42)
V, d = 10, 4
context_size = 2
batch_size = 3

W = np.random.randn(V, d) * 0.1
W_out = np.random.randn(d, V) * 0.1

# 构造上下文索引（每个样本 context_size 个词）
context_indices = [[1, 3], [0, 5], [7, 2]]   # 形状 (batch, context_size)
center_index = [2, 3, 1]                     # 目标词

loss = cbow_forward(context_indices, center_index, W, W_out)
print("CBOW loss:", loss)

CBOW loss: 2.297804812927644


## 6 注意力机制

### 6.1 理论计算题

给定 $Q \in \mathbb{R}^{2 \times 4}$，$K \in \mathbb{R}^{3 \times 4}$，$V \in \mathbb{R}^{3 \times 5}$，$d_k = 4$，缩放点积注意力：

**步骤 1**：计算得分矩阵 $S = Q K^\top / \sqrt{d_k}$  
（由于未给出具体数值，我们以符号表示，但可以列出形式）

设 $Q = \begin{bmatrix} q_1 \\ q_2 \end{bmatrix}$，$K = \begin{bmatrix} k_1 \\ k_2 \\ k_3 \end{bmatrix}$，则 $S_{ij} = \frac{q_i \cdot k_j}{\sqrt{4}} = \frac{q_i \cdot k_j}{2}$。

**步骤 2**：对 $S$ 的每一行（每个 query）做 softmax，得到注意力权重矩阵 $A \in \mathbb{R}^{2 \times 3}$：
$$
A_{ij} = \frac{\exp(S_{ij})}{\sum_{l=1}^{3} \exp(S_{il})}
$$

**步骤 3**：输出 $O = A \cdot V$，其中 $O \in \mathbb{R}^{2 \times 5}$，$O_i = \sum_{j=1}^{3} A_{ij} \cdot V_j$（$V_j$ 为第 $j$ 个 value 向量，维度 5）。

### 6.2 编程题

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch, _ = X.size()
        
        # 线性投影
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 拆分为多头： (seq_len, batch, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        # 使用 reshape 确保连续
        Q = Q.reshape(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        K = K.reshape(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        V = V.reshape(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        
        # 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)  # (batch, num_heads, seq_len, seq_len)
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)  # (batch, num_heads, seq_len, d_k)
        
        # 合并头： (batch, seq_len, num_heads*d_k) -> (seq_len, batch, d_model)
        attn_output = attn_output.permute(2, 0, 1, 3).reshape(seq_len, batch, self.d_model)
        
        # 最终线性层
        output = self.W_o(attn_output)
        return output  # (seq_len, batch, d_model)

In [9]:
# 测试 MultiHeadAttention
torch.manual_seed(42)
d_model, num_heads = 4, 2
seq_len, batch = 5, 2
X = torch.randn(seq_len, batch, d_model)

model = MultiHeadAttention(d_model, num_heads)
output = model(X)

print("Output shape:", output.shape)
print("Output first time step, first sample:\n", output[0, 0, :])

Output shape: torch.Size([5, 2, 4])
Output first time step, first sample:
 tensor([ 0.0018, -0.4840, -0.1516, -0.2003], grad_fn=<SliceBackward0>)
